# Synthetic corpus v2 — SD3 + ControlNet Depth, streamed to the Hugging Face Hub

**Stage:** 2 · Synthesis (after the defense) · **Status:** in progress (post-defense) · **Runtime:** Kaggle, 1× T4 (~16 GB), internet

Follow-up to the thesis. It addresses the synthetic-to-real gap with a different control signal: a **pseudo depth map** in which rune strokes are blurred grooves in a gently undulating stone surface. This replaces the binary Canny edges and drives `InstantX/SD3-Controlnet-Depth`. Every batch of 100 images is uploaded to the HF dataset `AntoniusPerf/runic-synth-sd3-depth`, so long generations survive Kaggle session limits.

| | |
|---|---|
| **Inputs** | same lexicon datasets as `02_synth_sd3_canny_final.ipynb` |
| **Outputs** | HF dataset `AntoniusPerf/runic-synth-sd3-depth` (`shard_<id>/batch_*.zip`, `labels_*.csv`) |

> `HF_TOKEN` needs **write** access (uploads) and an accepted SD3 Medium license.

## 1. Setup

In [ ]:
# !pip -q install --upgrade diffusers>=0.29 transformers>=4.40 accelerate safetensors \
#                           sentencepiece opencv-python-headless pandas numpy pillow tqdm
# !pip -q install --upgrade "Pillow>=11.0,<12"

In [2]:
import os
import gc
import re
import time
import zipfile
import urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import torch
from PIL import Image, ImageDraw, ImageFont
from tqdm.auto import tqdm
from huggingface_hub import HfApi

WORK = Path("/kaggle/working")
WORK.mkdir(parents=True, exist_ok=True)
print("Working dir:", WORK, "| CUDA:", torch.cuda.is_available())

# HF-токен из Kaggle Secrets (Add-ons → Secrets, ключ HF_TOKEN).
# ВАЖНО: токен должен иметь право WRITE — он используется и для скачивания SD3,
# и для загрузки zip-батчей в датасет на HF Hub. Лицензия SD3 Medium должна быть принята.
HF_TOKEN = os.environ.get("HF_TOKEN")  # never hard-code tokens in the notebook
if not HF_TOKEN:
    try:  # Kaggle: Add-ons -> Secrets -> HF_TOKEN
        from kaggle_secrets import UserSecretsClient

        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN = None

Working dir: /kaggle/working | CUDA: True


## 2. Lexicon: transliterations → words → runes
Words are tokenized from the transliteration columns, filtered by the alphabet Σ (3–25 characters,
frequency ≥ 2) and mapped to Unicode runes. The `assert` checks that `runes → transliteration` is lossless
for the whole lexicon.

In [3]:
# ── ЯЧЕЙКА 2. КОРПУС: CSV → чистка → руны ──────────────────────────────
# Укажите имя вашего Kaggle Dataset с CSV-файлами.
# После Add data они окажутся в /kaggle/input/<dataset-slug>/.
INPUT_DIR = Path("/kaggle/input")  # сканируем подключённые датасеты
CSV_NAMES = [
    "datasets/zhopa228/christerhamp-gamla-runor-with-filenames/christerhamp_gamla_runor_with_filenames.csv",
    "datasets/zhopa228/runic-inscriptions/runic_inscriptions_20251228_140136.csv",
    "datasets/zhopa228/runer-ku/runer_ku_20260114_234216.csv",
]


def find_csvs():
    found = {}
    for name in CSV_NAMES:
        for p in INPUT_DIR.rglob(name):
            found[name] = p
            break
    if len(found) < len(CSV_NAMES):
        # fallback: возможно, CSV лежат в /kaggle/working (например, загружены вручную)
        for name in CSV_NAMES:
            p = WORK / name
            if p.exists():
                found[name] = p
    missing = [n for n in CSV_NAMES if n not in found]
    assert not missing, (
        f"Не найдены CSV: {missing}. Подключите датасет (Add data) или положите файлы в /kaggle/working."
    )
    return [found[n] for n in CSV_NAMES]


CSV_PATHS = find_csvs()
print("CSV входы:")
[print(" ", p) for p in CSV_PATHS]

TRANSLIT_TO_RUNE = {
    "f": "\u16a0",
    "u": "\u16a2",
    "þ": "\u16a6",
    "ą": "\u16ac",
    "r": "\u16b1",
    "k": "\u16b4",
    "h": "\u16bc",
    "n": "\u16be",
    "i": "\u16c1",
    "a": "\u16c5",
    "s": "\u16cb",
    "t": "\u16cf",
    "b": "\u16d2",
    "m": "\u16d8",
    "l": "\u16da",
    "R": "\u16e6",
    "e": "\u16c2",
    "g": "\u16b5",
    "d": "\u16d1",
    "p": "\u16d4",
    "y": "\u16a4",
    "o": "\u16ae",
    "c": "\u16cd",
    "z": "\u16ce",
    "v": "\u16a1",
    "ʀ": "\u16e7",
    "w": "\u16b9",
    "j": "\u16c3",
    "ð": "\u16a7",
    "ï": "\u16c7",
    "ŋ": "\u16dc",
    "æ": "\u16ab",
    "ø": "\u16af",
}
RUNE_TO_TRANSLIT = {v: k for k, v in TRANSLIT_TO_RUNE.items()}
VALID = set(TRANSLIT_TO_RUNE)
_SPLIT = re.compile("[^" + re.escape("".join(VALID)) + "]+")


def tokenize_words(s, min_len=3, max_len=25):
    if not isinstance(s, str) or not s:
        return []
    s = s.replace("A", "a").replace("\u1d00", "a")
    out = []
    for tok in _SPLIT.split(s):
        if tok and min_len <= len(tok) <= max_len and all(c in VALID for c in tok):
            out.append(tok)
    return out


def word_to_runes(w):
    return "".join(TRANSLIT_TO_RUNE[c] for c in w)


def build_corpus(paths, col="transliteration", min_len=3, max_len=25, min_count=2):
    from collections import Counter

    cnt = Counter()
    for p in paths:
        df = pd.read_csv(p)
        for s in df[col].dropna().astype(str):
            cnt.update(tokenize_words(s, min_len, max_len))
    rows = [
        {"word": w, "runic": word_to_runes(w), "count": c, "length": len(w)}
        for w, c in cnt.items()
        if c >= min_count
    ]
    return pd.DataFrame(rows).sort_values("count", ascending=False).reset_index(drop=True)


corpus = build_corpus(CSV_PATHS, min_len=3, max_len=25, min_count=2)
assert all("".join(RUNE_TO_TRANSLIT[c] for c in r) == w for w, r in zip(corpus.word, corpus.runic))
corpus.to_csv(WORK / "runic_corpus.csv", index=False, encoding="utf-8")
print(f"Корпус: {len(corpus)} слов (round-trip OK)")
print(corpus.head(8).to_string(index=False))

CSV входы:
  /kaggle/input/datasets/zhopa228/christerhamp-gamla-runor-with-filenames/christerhamp_gamla_runor_with_filenames.csv
  /kaggle/input/datasets/zhopa228/runic-inscriptions/runic_inscriptions_20251228_140136.csv
  /kaggle/input/datasets/zhopa228/runer-ku/runer_ku_20260114_234216.csv
Корпус: 8562 слов (round-trip OK)
 word runic  count  length
  sin   ᛋᛁᚾ   2586       3
  auk   ᛅᚢᚴ   2149       3
faþur ᚠᛅᚦᚢᚱ   1094       5
stain ᛋᛏᛅᛁᚾ    952       5
 stin  ᛋᛏᛁᚾ    914       4
  sun   ᛋᚢᚾ    845       3
  lit   ᛚᛁᛏ    799       3
  ide   ᛁᛑᛂ    653       3


## 3. Sampler, renderer and depth control map
`depth_control` follows the depth-ControlNet convention (brighter = closer). The stone surface is the bright foreground and
rune strokes are dark grooves. Gaussian blur imitates sloped groove walls, and three octaves of low-frequency noise add relief.

In [4]:
# ── ЯЧЕЙКА 3. СЭМПЛЕР + РЕНДЕР (авто-кегль, перенос) + Depth ────────────
DIVIDER_RUNE = "\u16ec"  # ᛬


class CorpusSampler:
    def __init__(self, df, temperature=0.7):
        self.words = df.word.to_numpy()
        self.runes = df.runic.to_numpy()
        c = df["count"].to_numpy().astype(float)
        w = np.ones_like(c) if temperature <= 0 else np.power(c, temperature)
        self.p = w / w.sum()

    def word(self, rng):
        i = int(rng.choice(len(self.words), p=self.p))
        return str(self.runes[i]), str(self.words[i])

    def phrase(self, rng, min_words=1, max_words=4):
        n = int(rng.integers(min_words, max_words + 1))
        R, T = [], []
        for _ in range(n):
            r, t = self.word(rng)
            R.append(r)
            T.append(t)
        return (DIVIDER_RUNE.join(R) if n > 1 else R[0]), " ".join(T)


_FONT_URL = (
    "https://github.com/notofonts/NotoSansRunic/raw/refs/heads/main"
    "/fonts/ttf/unhinted/instance_ttf/NotoSansRunic-Regular.ttf"
)
FONT_PATH = str(WORK / "NotoSansRunic-Regular.ttf")
if not Path(FONT_PATH).exists():
    urllib.request.urlretrieve(_FONT_URL, FONT_PATH)


def _layout(draw, text, req, canvas, margin, floor=28):
    usable = canvas - 2 * margin
    for fs in range(req, floor - 1, -4):
        f = ImageFont.truetype(FONT_PATH, fs)
        if draw.textbbox((0, 0), text, font=f)[2] <= usable:
            return f, [text]
    f = ImageFont.truetype(FONT_PATH, floor)
    lines, cur = [], ""
    for wd in text.split(DIVIDER_RUNE):
        trial = (cur + DIVIDER_RUNE + wd) if cur else wd
        if draw.textbbox((0, 0), trial, font=f)[2] <= usable:
            cur = trial
        else:
            if cur:
                lines.append(cur)
            cur = wd
    if cur:
        lines.append(cur)
    return f, lines


def render_runes(text, canvas=1024, req_size=140, margin=44, rng=None):
    base = Image.new("RGB", (canvas, canvas), (255, 255, 255))
    d = ImageDraw.Draw(base)
    font, lines = _layout(d, text, req_size, canvas, margin)
    asc, desc = font.getmetrics()
    lh = int((asc + desc) * 1.15)
    y0 = max(margin, (canvas - lh * len(lines)) // 2)
    for i, ln in enumerate(lines):
        bb = d.textbbox((0, 0), ln, font=font)
        x = (canvas - (bb[2] - bb[0])) // 2
        jx = int(rng.integers(-10, 10)) if rng is not None else 0
        d.text((max(margin, x + jx), y0 + i * lh - bb[1]), ln, fill=(0, 0, 0), font=font)
    return base


def depth_control(base, sigma=6.0, fg=45, bg=205, relief=12, rng=None):
    """Псевдо-карта глубины из рендера (конвенция depth-ControlNet: светлее = ближе).
    Поверхность камня — передний план (bg, светлая), штрихи рун — борозды (fg, тёмные).
    GaussianBlur(sigma) имитирует наклонные стенки борозды; низкочастотный шум relief —
    плавный рельеф поверхности, чтобы карта не была идеальной плоскостью."""
    g = np.array(base.convert("L")).astype(np.float32)
    depth = np.where(g < 128, float(fg), float(bg))  # штрих руны → глубже/темнее
    depth = cv2.GaussianBlur(depth, (0, 0), sigma)
    if relief > 0 and rng is not None:
        h, w = depth.shape
        for sc in (6, 12, 24):  # три октавы волнистости камня
            hs = max(2, h // sc)
            n = rng.uniform(-1.0, 1.0, (hs, hs)).astype(np.float32)
            depth += cv2.resize(n, (w, h), interpolation=cv2.INTER_CUBIC) * (relief / 3.0)
    depth = depth.clip(0, 255).astype(np.uint8)
    return Image.fromarray(cv2.cvtColor(depth, cv2.COLOR_GRAY2RGB))


def pil_stone(size, rng):
    t = rng.uniform(0.45, 0.70, (size, size))
    for sc in (4, 8, 16, 32):
        hs = max(1, size // sc)
        n = rng.uniform(-0.06, 0.06, (hs, hs))
        ni = Image.fromarray(((n + 0.5) * 255).clip(0, 255).astype(np.uint8)).resize(
            (size, size), Image.NEAREST
        )
        t += (np.array(ni).astype(float) / 255 - 0.5) * 0.04
    g = (t * int(rng.integers(140, 195))).clip(0, 255).astype(np.uint8)
    return Image.fromarray(np.stack([g, g, (g * 0.95).astype(np.uint8)], 2))


print("Рендер/сэмплер/Depth готовы.")

Рендер/сэмплер/Depth готовы.


## 4. SD3 + ControlNet Depth pipeline

In [5]:
# ── ЯЧЕЙКА 4. ПАЙПЛАЙН SD3 + ControlNet Depth ──────────────────────────
POSITIVE = (
    "ancient runic inscription carved into dark grey granite, deep incised runes, "
    "weathered rough eroded surface, even diffuse daylight, monochrome grey, "
    "archaeological macro photo, sharp focus, realistic stone texture, no fantasy, no glow"
)
NEGATIVE = (
    "smooth polished, modern font, fantasy, glow, blur, watermark, extra symbols, "
    "duplicated runes, cartoon, bright colors, illustration, text overlay"
)


def load_pipe(token):
    if not torch.cuda.is_available():
        return None, "pil_only"
    try:
        from diffusers import SD3ControlNetModel, StableDiffusion3ControlNetPipeline

        dtype = torch.float16
        cn = SD3ControlNetModel.from_pretrained(
            "InstantX/SD3-Controlnet-Depth", torch_dtype=dtype, token=token
        )
        p = StableDiffusion3ControlNetPipeline.from_pretrained(
            "stabilityai/stable-diffusion-3-medium-diffusers",
            controlnet=cn,
            torch_dtype=dtype,
            text_encoder_3=None,
            tokenizer_3=None,
            token=token,
        )
        p.enable_model_cpu_offload()
        return p, "sd3_depth"
    except Exception as e:
        print("SD3 не загрузился → PIL fallback:", e)
        return None, "pil_only"


def generate_one(text, pipe, mode, rng, idx, cfg):
    base = render_runes(text, cfg["canvas"], cfg["req_size"], cfg["margin"], rng)
    if mode == "sd3_depth":
        ctrl = depth_control(
            base, cfg["depth_sigma"], cfg["depth_fg"], cfg["depth_bg"], cfg["depth_relief"], rng
        )
        g = torch.Generator("cuda").manual_seed(cfg["seed"] + idx)
        return pipe(
            prompt=POSITIVE,
            negative_prompt=NEGATIVE,
            control_image=ctrl,
            num_inference_steps=cfg["steps"],
            guidance_scale=cfg["cfg_scale"],
            controlnet_conditioning_scale=cfg["controlnet_scale"],
            generator=g,
            width=cfg["canvas"],
            height=cfg["canvas"],
        ).images[0]
    stone = pil_stone(cfg["canvas"], rng)
    m = (np.array(base.convert("L")) > 200).astype(float)
    return Image.fromarray(
        (
            np.array(stone).astype(float) * m[..., None]
            + np.array(base).astype(float) * (1 - m[..., None])
        )
        .clip(0, 255)
        .astype(np.uint8)
    )


print("Пайплайн готов.")

Пайплайн готов.


## 5. Generation → zip batches → HF Hub (with retries and resume from the Hub state)

In [ ]:
# ── ЯЧЕЙКА 5. ЗАПУСК: PNG → zip-батчи по 75 → upload на HF Hub ──────────
CFG = dict(
    n_samples=2000,
    canvas=1024,
    req_size=140,
    margin=44,
    min_words=1,
    max_words=4,
    steps=28,
    cfg_scale=7.0,
    controlnet_scale=0.75,  # для depth обычно 0.5–0.9; подобрать визуально
    depth_sigma=6.0,
    depth_fg=45,
    depth_bg=205,
    depth_relief=12,
    temperature=0.7,
    seed=10001,
    batch_size=100,
    delete_local_after_zip=True,
    # ---- HuggingFace Hub ----
    hf_upload=True,
    hf_repo_name="runic-synth-sd3-depth",  # либо сразу 'user/repo'
    hf_private=True,
    delete_zip_after_upload=False,  # True — экономит место в /kaggle/working
    hf_retries=3,
    hf_retry_wait=15,
)

WORKER_ID = 0
# CFG['seed']=int(np.random.SeedSequence(1001).spawn(4)[WORKER_ID].generate_state(1)[0])

TMP_IMG = WORK / "_tmp_images"
TMP_IMG.mkdir(parents=True, exist_ok=True)
OUT = WORK / "dataset" / f"shard_{WORKER_ID}"
OUT.mkdir(parents=True, exist_ok=True)
LABELS = OUT / "labels_all.csv"
HF_PREFIX = f"shard_{WORKER_ID}"


class HfZipUploader:
    """Грузит zip на HF Hub с повторами; при неудаче zip остаётся локально."""

    def __init__(self, api, repo_id, prefix, delete_after=False, retries=3, wait=15):
        self.api = api
        self.repo_id = repo_id
        self.prefix = prefix
        self.delete_after = delete_after
        self.retries = retries
        self.wait = wait
        self.failed = []

    def __call__(self, zip_path):
        zip_path = Path(zip_path)
        for k in range(self.retries):
            try:
                self.api.upload_file(
                    path_or_fileobj=str(zip_path),
                    path_in_repo=f"{self.prefix}/{zip_path.name}",
                    repo_id=self.repo_id,
                    repo_type="dataset",
                )
                print(f"  ↑ HF: {self.prefix}/{zip_path.name}")
                if self.delete_after:
                    zip_path.unlink(missing_ok=True)
                return True
            except Exception as e:
                print(f"  upload {zip_path.name}, попытка {k + 1}/{self.retries}: {e}")
                time.sleep(self.wait)
        self.failed.append(zip_path)
        print("  !! не загружен (остался локально):", zip_path.name)
        return False

    def retry_failed(self):
        rest, self.failed = self.failed, []
        for p in rest:
            if Path(p).exists():
                self(p)


def hub_list_batches(api, repo_id, prefix):
    try:
        return sorted(
            f
            for f in api.list_repo_files(repo_id, repo_type="dataset")
            if f.startswith(prefix + "/batch_") and f.endswith(".zip")
        )
    except Exception as e:
        print("list_repo_files:", e)
        return []


# ---- HF: репозиторий-датасет, доза-грузка локальных zip, учёт уже загруженного
api = HfApi(token=HF_TOKEN)
uploader = None
hub_batches = []
if CFG["hf_upload"]:
    REPO_ID = (
        CFG["hf_repo_name"]
        if "/" in CFG["hf_repo_name"]
        else f"{api.whoami()['name']}/{CFG['hf_repo_name']}"
    )
    api.create_repo(REPO_ID, repo_type="dataset", private=CFG["hf_private"], exist_ok=True)
    uploader = HfZipUploader(
        api,
        REPO_ID,
        HF_PREFIX,
        CFG["delete_zip_after_upload"],
        CFG["hf_retries"],
        CFG["hf_retry_wait"],
    )
    hub_batches = hub_list_batches(api, REPO_ID, HF_PREFIX)
    have = {Path(f).name for f in hub_batches}
    for z in sorted(OUT.glob("batch_*.zip")):  # локальные zip, не доехавшие до Hub
        if z.name not in have:
            uploader(z)
    hub_batches = hub_list_batches(api, REPO_ID, HF_PREFIX)
    print(f"HF repo: {REPO_ID} | на Hub уже {len(hub_batches)} батчей ({HF_PREFIX}/)")


class BatchZipper:
    """Каждые batch_size образцов → zip в /kaggle/working; через on_zip уходит на Hub."""

    def __init__(self, img_dir, out_dir, batch_size, delete_after, on_zip=None, start_idx=0):
        self.img_dir = Path(img_dir)
        self.out_dir = Path(out_dir)
        self.batch_size = batch_size
        self.delete_after = delete_after
        self.on_zip = on_zip
        self.pending = []
        self.batch_idx = start_idx
        ex = sorted(self.out_dir.glob("batch_*.zip"))
        if ex:
            self.batch_idx = max(self.batch_idx, int(ex[-1].stem.split("_")[1]) + 1)

    def add(self, rec):
        self.pending.append(rec)
        if len(self.pending) >= self.batch_size:
            self.flush()

    def flush(self):
        if not self.pending:
            return
        self.out_dir.mkdir(parents=True, exist_ok=True)
        zip_path = self.out_dir / f"batch_{self.batch_idx:04d}.zip"
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            for r in self.pending:
                p = self.img_dir / r["filename"]
                if p.exists():
                    zf.write(p, f"images/{r['filename']}")
            zf.writestr(
                f"labels_{self.batch_idx:04d}.csv", pd.DataFrame(self.pending).to_csv(index=False)
            )
        if self.delete_after:
            for r in self.pending:
                p = self.img_dir / r["filename"]
                if p.exists():
                    p.unlink()
        size_mb = zip_path.stat().st_size / 1e6
        print(f"  batch_{self.batch_idx:04d}.zip готов ({len(self.pending)} шт, {size_mb:.1f} MB)")
        if self.on_zip:
            self.on_zip(zip_path)
        self.batch_idx += 1
        self.pending.clear()


sampler = CorpusSampler(corpus, temperature=CFG["temperature"])
pipe, mode = load_pipe(HF_TOKEN)
print("Режим:", mode)
rng = np.random.default_rng(CFG["seed"])

# ---- resume: локальный labels_all.csv ИЛИ число батчей на Hub (после рестарта сессии)
records, start = [], 0
if LABELS.exists():
    try:
        ex = pd.read_csv(LABELS)
        start = len(ex)
        records = ex.to_dict("records")
    except Exception:
        start, records = 0, []
start = max(start, len(hub_batches) * CFG["batch_size"])
for _ in range(start):
    sampler.phrase(rng, CFG["min_words"], CFG["max_words"])  # синхронизация RNG
if start:
    print("Продолжение с", start)

zipper = BatchZipper(
    TMP_IMG,
    OUT,
    CFG["batch_size"],
    CFG["delete_local_after_zip"],
    on_zip=uploader,
    start_idx=len(hub_batches),
)


def safe(t):
    return t.replace("/", "-").replace("\\", "-").replace(" ", "_")


try:
    for i in tqdm(range(start, CFG["n_samples"]), initial=start, total=CFG["n_samples"]):
        runic, translit = sampler.phrase(rng, CFG["min_words"], CFG["max_words"])
        fn = f"syn_w{WORKER_ID}_{i:06d}_{safe(translit)}_{safe(runic)}.png"
        try:
            img = generate_one(runic, pipe, mode, rng, i, CFG)
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                torch.cuda.empty_cache()
                gc.collect()
                continue
            raise
        img.save(TMP_IMG / fn)
        rec = {"filename": fn, "runic": runic, "translit": translit}
        records.append(rec)
        zipper.add(rec)
        if (i + 1) % CFG["batch_size"] == 0:
            pd.DataFrame(records).to_csv(LABELS, index=False, encoding="utf-8")
finally:
    zipper.flush()
    pd.DataFrame(records).to_csv(LABELS, index=False, encoding="utf-8")
    if uploader:
        uploader.retry_failed()
        try:
            api.upload_file(
                path_or_fileobj=str(LABELS),
                path_in_repo=f"{HF_PREFIX}/labels_{time.strftime('%Y%m%d_%H%M%S')}.csv",
                repo_id=REPO_ID,
                repo_type="dataset",
            )
            print("  ↑ HF: labels_all.csv этой сессии загружен")
        except Exception as e:
            print("labels upload:", e)
    print(f"\nГотово: {len(records)} записей в сессии → {OUT} ({zipper.batch_idx} батчей всего)")
    if uploader:
        print(f"Датасет на Hub: https://huggingface.co/datasets/{REPO_ID} (папка {HF_PREFIX}/)")

HF repo: AntoniusPerf/runic-synth-sd3-depth | на Hub уже 4 батчей (shard_0/)
Режим: sd3_depth
Продолжение с 400
  batch_0004.zip готов (100 шт, 211.2 MB)
  ↑ HF: shard_0/batch_0004.zip
  batch_0005.zip готов (100 шт, 210.0 MB)
  ↑ HF: shard_0/batch_0005.zip
  batch_0006.zip готов (100 шт, 210.2 MB)
  ↑ HF: shard_0/batch_0006.zip
  batch_0007.zip готов (100 шт, 209.4 MB)
  ↑ HF: shard_0/batch_0007.zip
  batch_0008.zip готов (100 шт, 212.2 MB)
  ↑ HF: shard_0/batch_0008.zip
  batch_0009.zip готов (100 шт, 211.9 MB)
  ↑ HF: shard_0/batch_0009.zip
